# 01. Base Analitica de Empresas

## Goal
Define the analytical grain and rebuild both raw sources around the validated company key or RUC.


## Inputs
- Raw Excel files from `01_data_ingestion_enrichment/`
- `02_data_cleaning/outputs/match_final_empresas.csv`

## Outputs
- `outputs/base_empresas_analitica.parquet`
- `outputs/base_empresas_analitica.csv`


In [ ]:

# ── Helpers y rutas ──────────────────────────────────────────────────────────
from pathlib import Path
import re, unicodedata
import numpy as np
import pandas as pd

try:
    from IPython.display import display
except Exception:
    def display(x): print(x)

def find_project_root(start=None):
    start = (start or Path.cwd()).resolve()
    for p in [start, *start.parents]:
        if (p / "01_data_ingestion_enrichment").is_dir() and (p / "02_data_cleaning").is_dir():
            return p
    raise FileNotFoundError(f"No se pudo localizar la raíz.")

def ensure_dir(d): Path(d).mkdir(parents=True, exist_ok=True); return Path(d)

def save_df_csv(df, path, *, index=False, encoding="utf-8-sig"):
    path = Path(path); ensure_dir(path.parent)
    df.to_csv(path, index=index, encoding=encoding)
    print(f"[OK] Guardado: {path.resolve()}  shape: {df.shape}")
    return path

def read_csv_checked(path, **kwargs):
    path = Path(path)
    if not path.exists(): raise FileNotFoundError(f"No existe: {path.resolve()}")
    return pd.read_csv(path, **kwargs)

def read_excel_checked(path, **kwargs):
    path = Path(path)
    if not path.exists(): raise FileNotFoundError(f"No existe: {path.resolve()}")
    return pd.read_excel(path, **kwargs)

ROOT         = find_project_root()
INGESTION    = ROOT / "01_data_ingestion_enrichment"
CLEANING_OUT = ROOT / "02_data_cleaning" / "outputs"
OUTPUT_DIR   = ROOT / "03_feature_engineering" / "outputs"
ensure_dir(OUTPUT_DIR)

print(f"[CONFIG] ROOT        : {ROOT}")
print(f"[CONFIG] INGESTION   : {INGESTION}")
print(f"[CONFIG] CLEANING_OUT: {CLEANING_OUT}")
print(f"[CONFIG] OUTPUT_DIR  : {OUTPUT_DIR}")


## Build Plan
- Load the validated entity map.
- Link raw `Company` rows from leads to the validated entity key.
- Link raw `EMPRESA` rows from horas to the same validated entity key.
- Create one row per final company or RUC.
- Keep source presence flags to know whether the entity came from leads, horas, or both.


In [ ]:

# ── Construir base analítica ──────────────────────────────────────────────────
import pickle

# 1) Cargar mapa de entidades validadas
match_final = read_csv_checked(CLEANING_OUT / "match_final_empresas.csv")
print(f"[MATCH FINAL] {match_final.shape}  cols: {match_final.columns.tolist()}")

# 2) Cargar fuentes raw
_leads_path = next((p for p in [INGESTION/"leads.xlsx", ROOT/"leads.xlsx"] if p.exists()), None)
_horas_path = next((p for p in [INGESTION/"proyectos_empresa.xlsx", ROOT/"proyectos_empresa.xlsx"] if p.exists()), None)

if _leads_path is None: raise FileNotFoundError("No encontré leads.xlsx")
if _horas_path is None: raise FileNotFoundError("No encontré proyectos_empresa.xlsx")

df_leads = read_excel_checked(_leads_path)
df_horas  = read_excel_checked(_horas_path)
print(f"[LEADS] {df_leads.shape}")
print(f"[HORAS] {df_horas.shape}")

# 3) Normalización (igual a staging)
def _strip_accents(s):
    return "".join(ch for ch in unicodedata.normalize("NFKD", str(s)) if not unicodedata.combining(ch))

def normalize_company_name(value) -> str:
    if value is None or (isinstance(value, float) and np.isnan(value)): return ""
    s = _strip_accents(str(value).strip()).upper()
    s = re.sub(r"[^A-Z0-9 ]+", " ", s)
    s = re.sub(r"\b(SA|S\s*A|S\.A\.?|S\.A\.S\.?|SAS|LTDA|CIA|C\.?IA\.?|COMPANIA|COMPAÑIA|CORP|INC|LLC|C\.L\.?|C\.?LTDA\.?|\&|Y)\b"," ",s)
    s = re.sub(r"\b(DE|DEL|LA|EL|LOS|LAS)\b", " ", s)
    return re.sub(r"\s+", " ", s).strip()

df_leads["_name_norm"] = df_leads["Company"].map(normalize_company_name)
df_horas["_name_norm"]  = df_horas["EMPRESA"].map(normalize_company_name)

# 4) Construir base: una fila por entidad canónica
base_leads = (
    df_leads.groupby("_name_norm", as_index=False)
    .agg(n_rows_leads=("_name_norm","size"))
    .rename(columns={"_name_norm": "name_norm"})
)
base_horas = (
    df_horas.groupby("_name_norm", as_index=False)
    .agg(n_rows_horas=("_name_norm","size"))
    .rename(columns={"_name_norm": "name_norm"})
)

# Partir de match_final como "universo" de nombres
base = match_final[["name_norm","name_raw","RUC","source_winner","verdict"]].copy()
base = (
    base
    .merge(base_leads, on="name_norm", how="left")
    .merge(base_horas,  on="name_norm", how="left")
)
base["n_rows_leads"] = base["n_rows_leads"].fillna(0).astype(int)
base["n_rows_horas"]  = base["n_rows_horas"].fillna(0).astype(int)
base["source_in_leads"] = base["n_rows_leads"] > 0
base["source_in_horas"]  = base["n_rows_horas"]  > 0
base["entity_id"]    = ["ENT_{:04d}".format(i) for i in range(len(base))]
base["canonical_name"] = base["name_raw"].fillna(base["name_norm"])

# Columnas finales recomendadas
final_cols = ["entity_id","RUC","canonical_name","name_norm",
              "source_in_leads","source_in_horas","n_rows_leads","n_rows_horas",
              "source_winner","verdict"]
final_cols = [c for c in final_cols if c in base.columns]
base_analitica = base[final_cols].reset_index(drop=True)

print(f"\n[BASE ANALITICA] {base_analitica.shape}")
print(f"  En LEADS: {base_analitica['source_in_leads'].sum()}")
print(f"  En HORAS: {base_analitica['source_in_horas'].sum()}")
print(f"  En ambas: {(base_analitica['source_in_leads'] & base_analitica['source_in_horas']).sum()}")

_ = save_df_csv(base_analitica, OUTPUT_DIR / "base_empresas_analitica.csv")
with open(OUTPUT_DIR / "base_empresas_analitica.pkl", "wb") as f: pickle.dump(base_analitica, f)
print(f"[OK] base_empresas_analitica.pkl guardado")

# ── Verificación ──────────────────────────────────────────────────────────────
for p in [OUTPUT_DIR/"base_empresas_analitica.csv", OUTPUT_DIR/"base_empresas_analitica.pkl"]:
    if not p.exists(): raise FileNotFoundError(f"Output faltante: {p}")
    print(f"[OK] {p.name}")
print("\n✓ Notebook 01_base_analitica_empresas completado correctamente.")
display(base_analitica.head(10))
